# 🚀 TG Bot to Kaggle — 1-tap Deploy
Run each cell top-to-bottom. The app will be live in ~2 minutes.

**What this does:**
- Pulls the latest code from GitHub
- Builds the Docker image
- Starts the app and exposes it via a public URL using `ngrok`
- Registers that URL as your Telegram bot webhook automatically

In [ ]:
# ── Step 1: Set your secrets ─────────────────────────────────────────────────
import os

os.environ['KAGGLE_USERNAME']    = 'your_kaggle_username'   # ← fill in
os.environ['KAGGLE_KEY']         = 'your_kaggle_api_key'    # ← fill in
os.environ['TELEGRAM_BOT_TOKEN'] = 'your_telegram_bot_token' # ← fill in
os.environ['SESSION_SECRET']     = 'change_me_random_32chars' # ← fill in
os.environ['NGROK_AUTH_TOKEN']   = 'your_ngrok_authtoken'   # ← get free at ngrok.com

print('✅ Secrets set')

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────────────
!pip install pyngrok -q
!apt-get install -y docker.io > /dev/null 2>&1
!service docker start > /dev/null 2>&1
print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Pull code from GitHub ────────────────────────────────────────────
!git clone https://github.com/shan-test-project/tg-bot-to-kaggle.git /app 2>&1 | tail -3
%cd /app
print('✅ Code pulled')

In [ ]:
# ── Step 4: Build Docker image ───────────────────────────────────────────────
!docker build -t tg-kaggle-app . 2>&1 | tail -5
print('✅ Image built')

In [ ]:
# ── Step 5: Start the app ────────────────────────────────────────────────────
import subprocess, os

cmd = [
    'docker', 'run', '-d', '--rm',
    '-p', '8080:8080',
    '-e', f"KAGGLE_USERNAME={os.environ['KAGGLE_USERNAME']}",
    '-e', f"KAGGLE_KEY={os.environ['KAGGLE_KEY']}",
    '-e', f"TELEGRAM_BOT_TOKEN={os.environ['TELEGRAM_BOT_TOKEN']}",
    '-e', f"SESSION_SECRET={os.environ['SESSION_SECRET']}",
    '--name', 'tg-kaggle',
    'tg-kaggle-app'
]
result = subprocess.run(cmd, capture_output=True, text=True)
print('Container ID:', result.stdout.strip()[:12])
print('✅ App started')

In [ ]:
# ── Step 6: Expose via ngrok & register Telegram webhook ─────────────────────
import time, requests
from pyngrok import ngrok

ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])
time.sleep(3)  # wait for container to be ready

tunnel = ngrok.connect(8080, 'http')
public_url = tunnel.public_url
print(f'🌐 Public URL: {public_url}')

# Register webhook with Telegram
resp = requests.post(f'{public_url}/api/telegram/setup')
print('Webhook:', resp.json())

print(f'\n✅ All done! Open in Telegram or visit: {public_url}')